# 00 -- Environment & Model Load Verification

**Goal of this notebook:** confirm the Python environment is wired correctly and that the [Cisco Time Series Model 1.0](https://huggingface.co/cisco-ai/cisco-time-series-model-1.0) (`cisco-tsm` package) can be imported and loaded.

This is **not** the full experiment. It stops as soon as the model runs one tiny forecast, just to prove the whole chain (venv -> kernel -> package -> weights -> inference) works end to end.

## Step 1 -- Confirm the kernel is the project's venv

Jupyter can silently fall back to a different Python if the kernel isn't selected correctly. This cell prints the interpreter path and version so you can visually confirm it points at `ai-foundation-models-lab/.venv`, not a system Python or another project's env.

In [1]:
import sys, platform
from pathlib import Path

# Show only the path from .venv onward -- keeps the host machine's home
# directory and username out of committed notebook output.
exe_parts = Path(sys.executable).parts
shown_exe = str(Path(*exe_parts[exe_parts.index(".venv"):])) if ".venv" in exe_parts else sys.executable

print("Python executable:", shown_exe)
print("Python version   :", sys.version.split()[0])
print("Platform         :", platform.platform())

assert sys.version_info[:2] == (3, 11), "Expected Python 3.11 -- wrong kernel selected?"
expected_msg = "Interpreter is not this project's venv -- pick the 'Python 3.11 (ai-foundation-models-lab)' kernel."
assert ".venv" in sys.executable and "ai-foundation-models-lab" in sys.executable, expected_msg
print("\nOK: running inside the project venv on Python 3.11.")

Python executable: .venv/bin/python
Python version   : 3.11.15
Platform         : macOS-26.5-arm64-arm-64bit

OK: running inside the project venv on Python 3.11.


## Step 2 -- Confirm PyTorch sees Apple Silicon

On an M-series Mac, PyTorch accelerates via **MPS** (Metal Performance Shaders), not CUDA. `torch.cuda.is_available()` will correctly be `False` here -- that's expected, not an error. The Cisco model's own example code picks `backend="cpu"` whenever CUDA isn't available, so it will run on CPU by default even though MPS exists; for a 250M-parameter model doing single-series inference, CPU is already fast enough for this smoke test.

In [2]:
import torch

print("torch version                :", torch.__version__)
print("CUDA available               :", torch.cuda.is_available())
print("MPS (Apple Silicon) available:", torch.backends.mps.is_available())

torch version                : 2.13.0
CUDA available               : False
MPS (Apple Silicon) available: True


## Step 3 -- Import the Cisco Time Series Model package

`cisco-tsm` is the PyPI package published by the Cisco/Splunk team. It wraps a TimesFM-derived architecture. If this import fails, the venv/install is broken -- nothing below will work either.

In [3]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

from cisco_tsm import CiscoTsmMR, TimesFmHparams, TimesFmCheckpoint
import cisco_tsm
from pathlib import Path

mod_parts = Path(cisco_tsm.__file__).parts
shown_mod_path = str(Path(*mod_parts[mod_parts.index(".venv"):])) if ".venv" in mod_parts else cisco_tsm.__file__

print("cisco_tsm imported OK, module path:", shown_mod_path)

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


Loaded PyTorch TimesFM, likely because python version is 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)].
cisco_tsm imported OK, module path: .venv/lib/python3.11/site-packages/cisco_tsm/__init__.py


## Step 4 -- Load the model weights from Hugging Face

`TimesFmCheckpoint` points at the `cisco-ai/cisco-time-series-model-1.0` repo on Hugging Face and downloads the ~250M-parameter checkpoint on first run (cached under `~/.cache/huggingface` afterwards, so subsequent loads are fast/offline). `TimesFmHparams` mirrors the architecture Cisco trained with (25 decoder layers, 15 output quantiles, no absolute positional embedding since it uses RoPE instead). `CiscoTsmMR` is the actual model object -- constructing it both instantiates the network and loads the downloaded weights into it, which is the real "can it load" test.

In [4]:
hparams = TimesFmHparams(
    num_layers=25,
    use_positional_embedding=False,
    backend="gpu" if torch.cuda.is_available() else "cpu",
    quantiles=[0.01, 0.05, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95, 0.99],
)

checkpoint = TimesFmCheckpoint(huggingface_repo_id="cisco-ai/cisco-time-series-model-1.0")

model = CiscoTsmMR(hparams=hparams, checkpoint=checkpoint)
print("Model loaded OK:", type(model))

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 2788.53it/s]

Model loaded OK: <class 'cisco_tsm.cisco_tsm_mr.CiscoTsmMR'>


## Step 5 -- Minimal smoke-test forecast

One short synthetic series, one `forecast()` call. The point isn't forecast quality -- it's confirming the loaded model can actually run a forward pass and hand back the expected shapes (a `128`-point mean forecast plus 15 quantile arrays). If this cell runs without error, the environment is fully verified.

In [5]:
import numpy as np

rng = np.random.default_rng(42)
toy_series = 50 + 5 * np.sin(np.linspace(0, 20 * np.pi, 600)) + rng.normal(0, 1, 600)

forecast = model.forecast(toy_series.astype(np.float32), horizon_len=128)

mean_forecast = forecast[0]["mean"]
quantiles = forecast[0]["quantiles"]

print("mean_forecast shape:", mean_forecast.shape)
print("quantile levels    :", sorted(quantiles.keys()))
print("\nSetup verified: environment, kernel, package, weights, and inference all work.")

mean_forecast shape: (128,)
quantile levels    : ['0.01', '0.05', '0.1', '0.2', '0.25', '0.3', '0.4', '0.5', '0.6', '0.7', '0.75', '0.8', '0.9', '0.95', '0.99']

Setup verified: environment, kernel, package, weights, and inference all work.
